# Reproject Radar Coordinates via the Accessor

This notebook demonstrates the ``target_crs`` option now exposed on
``.xradar.georeference()`` for ``xarray.Dataset`` and ``xarray.DataTree``.

It lets you reproject radar ``x, y`` coordinates from the radar-native
Azimuthal Equidistant (AEQD) projection into any ``pyproj``-compatible CRS
in a single accessor call:

```python
radar = radar.xradar.georeference(target_crs=4326)
```

## Imports

In [ ]:
import cmweather  # noqa
import matplotlib.pyplot as plt
from open_radar_data import DATASETS

import xradar as xd

## Read a sample radar file

In [ ]:
filename = DATASETS.fetch("cfrad.20080604_002217_000_SPOL_v36_SUR.nc")
radar = xd.io.open_cfradial1_datatree(filename, first_dim="auto")
radar

## 1. Default georeferencing (AEQD, meters)

Without ``target_crs``, the accessor adds ``x``, ``y``, ``z`` in the
radar-native AEQD projection (distances in meters from the radar site).

In [ ]:
radar_aeqd = radar.xradar.georeference()
sweep = radar_aeqd["sweep_0"].to_dataset(inherit="all_coords")
crs = sweep.xradar.get_crs()
print(
    "projection:",
    crs.coordinate_operation.method_name if crs.coordinate_operation else crs.name,
)
print("is_projected / is_geographic:", crs.is_projected, "/", crs.is_geographic)
print("x units:", sweep.x.attrs.get("units"))
print("x range:", float(sweep.x.min()), float(sweep.x.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
radar_aeqd["sweep_0"]["DBZ"].plot(x="x", y="y", cmap="ChaseSpectral", ax=ax)
ax.set_title("AEQD (default) — x, y in meters from radar")
ax.set_aspect("equal")

## 2. Reproject to geographic lon/lat (EPSG:4326)

Pass ``target_crs=4326`` to get ``x`` as longitude and ``y`` as latitude.
Attributes are updated automatically (``standard_name``, ``units``).

In [ ]:
radar_geo = radar.xradar.georeference(target_crs=4326)
sweep = radar_geo["sweep_0"].to_dataset(inherit="all_coords")
crs = sweep.xradar.get_crs()
print("CRS name:", crs.name, "| EPSG:", crs.to_epsg())
print("x attrs:", dict(sweep.x.attrs))
print("y attrs:", dict(sweep.y.attrs))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
radar_geo["sweep_0"]["DBZ"].plot(x="x", y="y", cmap="ChaseSpectral", ax=ax)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("EPSG:4326 — x=lon, y=lat")

## 3. Reproject to a projected CRS (UTM, Web Mercator)

Any ``pyproj``-compatible CRS input works: EPSG code, WKT string, or a
``pyproj.CRS`` instance.

In [ ]:
# Web Mercator (meters)
radar_wm = radar.xradar.georeference(target_crs="EPSG:3857")
sweep_wm = radar_wm["sweep_0"].to_dataset(inherit="all_coords")
crs = sweep_wm.xradar.get_crs()
print("CRS:", crs.name, "| EPSG:", crs.to_epsg())

In [ ]:
import pyproj

utm_crs = pyproj.CRS("EPSG:32633")  # UTM zone 33N (tweak for your radar site)
radar_utm = radar.xradar.georeference(target_crs=utm_crs)
sweep_utm = radar_utm["sweep_0"].to_dataset(inherit="all_coords")
crs = sweep_utm.xradar.get_crs()
print("CRS:", crs.name, "| EPSG:", crs.to_epsg())

## 4. Plot on a cartopy map

Once data is in a known CRS, hand it off to cartopy for map plotting.

In [ ]:
import cartopy.crs as ccrs

radar_geo = radar.xradar.georeference(target_crs=4326)

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
radar_geo["sweep_0"]["DBZ"].plot(
    x="x",
    y="y",
    cmap="ChaseSpectral",
    transform=ccrs.PlateCarree(),
    ax=ax,
    cbar_kwargs=dict(pad=0.05, shrink=0.7),
)
ax.coastlines()
ax.gridlines(draw_labels=True)
ax.set_title("Georeferenced PPI on PlateCarree map")

## 5. Works on a single Dataset too

The accessor is available on ``Dataset`` and ``DataArray`` as well, not just
``DataTree``.

In [ ]:
ds = radar["sweep_0"].to_dataset(inherit="all_coords")
ds_geo = ds.xradar.georeference(target_crs=4326)
ds_geo[["DBZ"]]

## Notes

* ``target_crs`` accepts anything ``pyproj.CRS(...)`` accepts: int
  [EPSG codes](https://epsg.io/), ``"EPSG:xxxx"`` strings, WKT, or
  ``pyproj.CRS`` instances.
* The ``spatial_ref`` / ``crs_wkt`` coordinate is updated to reflect the
  target CRS — ``sweep.xradar.get_crs()`` will return it.
* Currently only ``x`` and ``y`` are reprojected; ``z`` stays as AEQD
  altitude above the radar site.
* If you want **separate** ``lon`` / ``lat`` / ``alt`` coordinates (without
  overwriting ``x, y``), see the [Assign_GeoCoords](Assign_GeoCoords.md) notebook.